In [1]:
# TENNIS ANALYTICS PROJECT
# MEMBER 1 - DATA COLLECTION AND PREPARATION

import pandas as pd
import json
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
import os

os.chdir(r"C:\Users\BR REDDY\OneDrive\Desktop\tennis_analytics")

print("Current Folder:", os.getcwd())
print("Available Files:", os.listdir("data"))

Current Folder: C:\Users\BR REDDY\OneDrive\Desktop\tennis_analytics
Available Files: ['categories.csv', 'competitions.csv', 'competitions.json', 'competitors.csv', 'competitor_rankings.csv', 'complexes.csv', 'complexes.json', 'double_competitors_rankings.json', 'validation_report.txt', 'venues.csv']


In [4]:
# LOAD JSON FILES

with open("data/competitions.json", "r", encoding="utf-8") as f:
    competitions_data = json.load(f)

with open("data/complexes.json", "r", encoding="utf-8") as f:
    complexes_data = json.load(f)

with open("data/double_competitors_rankings.json", "r", encoding="utf-8") as f:
    rankings_data = json.load(f)

print("All JSON files loaded successfully!")

All JSON files loaded successfully!


In [5]:
# EXPLORE JSON STRUCTURE

print("COMPETITIONS JSON")
print("Main Keys:", competitions_data.keys())
print("Total Competitions:", len(competitions_data["competitions"]))
print("First Record:")
print(competitions_data["competitions"][0])

print("\nCOMPLEXES JSON")
print("Main Keys:", complexes_data.keys())
print("Total Complexes:", len(complexes_data["complexes"]))
print("First Record:")
print(complexes_data["complexes"][0])

print("\nRANKINGS JSON")
print("Main Keys:", rankings_data.keys())
print("Total Ranking Groups:", len(rankings_data["rankings"]))
print("First Group:")
print(rankings_data["rankings"][0]["name"])

COMPETITIONS JSON
Main Keys: dict_keys(['generated_at', 'competitions'])
Total Competitions: 6699
First Record:
{'id': 'sr:competition:620', 'name': 'Hopman Cup', 'type': 'mixed', 'gender': 'mixed', 'category': {'id': 'sr:category:181', 'name': 'Hopman Cup'}}

COMPLEXES JSON
Main Keys: dict_keys(['generated_at', 'complexes'])
Total Complexes: 780
First Record:
{'id': 'sr:complex:705', 'name': 'Nacional', 'venues': [{'id': 'sr:venue:70045', 'name': 'Cancha Central', 'city_name': 'Santiago', 'city_id': 'sr:city:148', 'country_name': 'Chile', 'country_code': 'CHL', 'timezone': 'America/Santiago'}]}

RANKINGS JSON
Main Keys: dict_keys(['generated_at', 'rankings'])
Total Ranking Groups: 2
First Group:
ATP


In [6]:
# EXTRACT COMPETITIONS AND CATEGORIES

competitions = []
categories = {}

for item in competitions_data["competitions"]:

    category = item.get("category") or {}

    if category.get("id"):
        categories[category["id"]] = {
            "category_id": category.get("id"),
            "category_name": category.get("name")
        }

    competitions.append({
        "competition_id": item.get("id"),
        "competition_name": item.get("name"),
        "type": item.get("type"),
        "gender": item.get("gender"),
        "category_id": category.get("id")
    })

competitions_df = pd.DataFrame(competitions)
categories_df = pd.DataFrame(categories.values())

print("Competitions Data:", competitions_df.shape)
print("Categories Data:", categories_df.shape)

display(competitions_df.head())
display(categories_df.head())

Competitions Data: (6699, 5)
Categories Data: (18, 2)


,competition_id,competition_name,type,gender,category_id
0,sr:competition:620,Hopman Cup,mixed,mixed,sr:category:181
1,sr:competition:660,World Team Cup,mixed,men,sr:category:3
2,sr:competition:990,ATP Challenger Tour Finals,singles,men,sr:category:72
3,sr:competition:1207,Championship International Series,singles,women,sr:category:6
4,sr:competition:2100,Davis Cup,mixed,men,sr:category:76


,category_id,category_name
0,sr:category:181,Hopman Cup
1,sr:category:3,ATP
2,sr:category:72,Challenger
3,sr:category:6,WTA
4,sr:category:76,Davis Cup


In [7]:
# EXTRACT COMPLEXES AND VENUES

complexes = []
venues = []

for item in complexes_data["complexes"]:

    complexes.append({
        "complex_id": item.get("id"),
        "complex_name": item.get("name")
    })

    for venue in item.get("venues", []):

        venues.append({
            "venue_id": venue.get("id"),
            "venue_name": venue.get("name"),
            "complex_id": item.get("id"),
            "city_name": venue.get("city_name"),
            "city_id": venue.get("city_id"),
            "country_name": venue.get("country_name"),
            "country_code": venue.get("country_code"),
            "timezone": venue.get("timezone")
        })

complexes_df = pd.DataFrame(complexes)
venues_df = pd.DataFrame(venues)

print("Complexes Data:", complexes_df.shape)
print("Venues Data:", venues_df.shape)

display(complexes_df.head())
display(venues_df.head())

Complexes Data: (780, 2)
Venues Data: (4139, 8)


,complex_id,complex_name
0,sr:complex:705,Nacional
1,sr:complex:1078,Estadio de La Cartuja
2,sr:complex:1495,Sibur Arena
3,sr:complex:2375,Complexo de Tenis do Jamor
4,sr:complex:4032,Shree Shiv Chhatrapati Sports Complex


,venue_id,venue_name,complex_id,city_name,city_id,country_name,country_code,timezone
0,sr:venue:70045,Cancha Central,sr:complex:705,Santiago,sr:city:148,Chile,CHL,America/Santiago
1,sr:venue:74856,Centre Court,sr:complex:1078,Seville,sr:city:175,Spain,ESP,Europe/Madrid
2,sr:venue:74858,Court One,sr:complex:1078,Seville,sr:city:175,Spain,ESP,Europe/Madrid
3,sr:venue:1496,Court 1,sr:complex:1495,Saint Petersburg,sr:city:348,Russia,RUS,Europe/Moscow
4,sr:venue:1500,CENTER COURT,sr:complex:1495,Saint Petersburg,sr:city:348,Russia,RUS,Europe/Moscow


In [8]:
# EXTRACT COMPETITORS AND RANKINGS

competitors = {}
rankings = []

for group in rankings_data["rankings"]:

    for item in group.get("competitor_rankings", []):

        player = item.get("competitor") or {}
        player_id = player.get("id")

        if player_id:
            competitors[player_id] = {
                "competitor_id": player_id,
                "competitor_name": player.get("name"),
                "country": player.get("country"),
                "country_code": player.get("country_code"),
                "abbreviation": player.get("abbreviation")
            }

        rankings.append({
            "competitor_id": player_id,
            "rank": item.get("rank"),
            "movement": item.get("movement"),
            "points": item.get("points"),
            "competitions_played": item.get("competitions_played"),
            "ranking_type": group.get("name"),
            "gender": group.get("gender"),
            "year": group.get("year"),
            "week": group.get("week")
        })

competitors_df = pd.DataFrame(competitors.values())
rankings_df = pd.DataFrame(rankings)

print("Competitors Data:", competitors_df.shape)
print("Rankings Data:", rankings_df.shape)

display(competitors_df.head())
display(rankings_df.head())

Competitors Data: (1000, 5)
Rankings Data: (1000, 9)


,competitor_id,competitor_name,country,country_code,abbreviation
0,sr:competitor:14898,"Heliovaara, Harri",Finland,FIN,HEL
1,sr:competitor:637970,"Patten, Henry",Great Britain,GBR,PAT
2,sr:competitor:53785,"Skupski, Neal",Great Britain,GBR,SKU
3,sr:competitor:52065,"Harrison, Christian",USA,USA,HAR
4,sr:competitor:16160,"Zeballos, Horacio",Argentina,ARG,ZEB


,competitor_id,rank,movement,points,competitions_played,ranking_type,gender,year,week
0,sr:competitor:14898,1,0,10540,19,ATP,men,2026,42
1,sr:competitor:637970,1,0,10540,19,ATP,men,2026,42
2,sr:competitor:53785,3,0,6890,23,ATP,men,2026,42
3,sr:competitor:52065,4,0,6050,24,ATP,men,2026,42
4,sr:competitor:16160,5,0,6030,15,ATP,men,2026,42


In [9]:
# EXPORT DATAFRAMES TO CSV

os.makedirs("data", exist_ok=True)

competitions_df.to_csv("data/competitions.csv", index=False)
categories_df.to_csv("data/categories.csv", index=False)

complexes_df.to_csv("data/complexes.csv", index=False)
venues_df.to_csv("data/venues.csv", index=False)

competitors_df.to_csv("data/competitors.csv", index=False)
rankings_df.to_csv("data/competitor_rankings.csv", index=False)

print("All 6 CSV files exported successfully!")

All 6 CSV files exported successfully!


In [10]:
# DATA VALIDATION

csv_files = [
    "categories.csv",
    "competitions.csv",
    "competitors.csv",
    "competitor_rankings.csv",
    "complexes.csv",
    "venues.csv"
]

for file in csv_files:

    df = pd.read_csv("data/" + file)

    print("\nFile:", file)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicates:", df.duplicated().sum())
    print("Missing Values:", df.isnull().sum().sum())


File: categories.csv
Rows: 18
Columns: 2
Duplicates: 0
Missing Values: 0

File: competitions.csv
Rows: 6699
Columns: 5
Duplicates: 0
Missing Values: 0

File: competitors.csv
Rows: 1000
Columns: 5
Duplicates: 0
Missing Values: 50

File: competitor_rankings.csv
Rows: 1000
Columns: 9
Duplicates: 0
Missing Values: 0

File: complexes.csv
Rows: 780
Columns: 2
Duplicates: 0
Missing Values: 0

File: venues.csv
Rows: 4139
Columns: 8
Duplicates: 0
Missing Values: 0


In [11]:
# MEMBER 1 - FINAL SUMMARY

print("TENNIS ANALYTICS PROJECT")
print("=" * 40)

print("JSON Files Processed: 3")
print("CSV Files Generated: 6")

print("\nFINAL RECORD COUNTS")

print("Categories:", len(categories_df))
print("Competitions:", len(competitions_df))
print("Complexes:", len(complexes_df))
print("Venues:", len(venues_df))
print("Competitors:", len(competitors_df))
print("Competitor Rankings:", len(rankings_df))

print("\nDATA VALIDATION")
print("Duplicate Records: 0")
print("Missing Country Codes: 50")

print("\nMember 1 Data Preparation Completed Successfully!")

TENNIS ANALYTICS PROJECT
JSON Files Processed: 3
CSV Files Generated: 6

FINAL RECORD COUNTS
Categories: 18
Competitions: 6699
Complexes: 780
Venues: 4139
Competitors: 1000
Competitor Rankings: 1000

DATA VALIDATION
Duplicate Records: 0
Missing Country Codes: 50

Member 1 Data Preparation Completed Successfully!


In [12]:
import os

print("Notebook Working Folder:", os.getcwd())
print("IPython Files:", [f for f in os.listdir() if f.endswith(".ipynb")])


Notebook Working Folder: C:\Users\BR REDDY\OneDrive\Desktop\tennis_analytics
IPython Files: []
